In [8]:
import pandas as pd

pest_name = '탄저병'

files = [
    'AIM_PEX_DT03_샘플데이터/RDA_OBSERVATION_APPLE.csv',
    'AIM_PEX_DT03_샘플데이터/RDA_OBSERVATION_BEAN.csv',
    'AIM_PEX_DT03_샘플데이터/RDA_OBSERVATION_PEPPER.csv',
    'AIM_PEX_DT03_샘플데이터/RDA_OBSERVATION_RICE.csv'
]

meta_cols = ['조사년도', '지역-시도', '지역-시군구', '지역-읍면동',
             '상세주소', '좌표-경도', '좌표-위도', '조사회차', '조사일자(YYYYMMDD)',
             '조사구분', '품종명']

all_data = []
for file in files:
    df = pd.read_csv(file)
    pest_cols = [col for col in df.columns if pest_name in col]
    existing_meta = [col for col in meta_cols if col in df.columns]

    if pest_cols:
        selected = df[existing_meta + pest_cols].copy()
        selected = selected.dropna(subset=['좌표-경도', '좌표-위도'])
        all_data.append(selected)

result_df = pd.concat(all_data, ignore_index=True)

law_path = "/Users/doyoung-gil/Desktop/연구실/법정동코드_전체자료.txt"
law_df = pd.read_csv(law_path, sep='\t', encoding='cp949', dtype={'법정동코드': str})

law_df = law_df[law_df['폐지여부'] == '존재'].copy()
law_df[['시도', '시군구', '읍면동']] = law_df['법정동명'].str.split(' ', n=2, expand=True)

result_df['지역-시군구'] = result_df['지역-시군구'].replace({'청원군': '청주시'})

merged = pd.merge(
    result_df,
    law_df[['법정동코드', '시도', '시군구', '읍면동']],
    left_on=['지역-시도', '지역-시군구', '지역-읍면동'],
    right_on=['시도', '시군구', '읍면동'],
    how='left'
)

missing = merged[merged['법정동코드'].isna()].copy()
law_df_sgg = law_df[law_df['읍면동'].isna()]  # 읍면동 없는 법정동만

backup = pd.merge(
    missing.drop(columns=['법정동코드', '시도', '시군구', '읍면동']),
    law_df_sgg[['법정동코드', '시도', '시군구']],
    left_on=['지역-시도', '지역-시군구'],
    right_on=['시도', '시군구'],
    how='left'
)

success = merged[merged['법정동코드'].notna()]
final_df = pd.concat([success, backup], ignore_index=True)

final_df['법정동코드'] = final_df['법정동코드'].astype(str).str.zfill(10)

final_df.drop(columns=['시도', '시군구', '읍면동'], errors='ignore', inplace=True)
final_df.rename(columns={'법정동코드': '지점ID'}, inplace=True)

final_df = final_df.sort_values(by=['지점ID', '조사년도'])

pest_cols = [c for c in final_df.columns if pest_name in c]

if pest_cols:
    base_cols = [c for c in final_df.columns if c not in pest_cols and c != '지점ID']
    final_order = ['지점ID'] + base_cols + pest_cols
    final_df = final_df[final_order]
else:
    final_df = final_df[['지점ID'] + [c for c in final_df.columns if c != '지점ID']]

final_df.to_csv(f"{pest_name}_sample_1.csv", index=False, encoding='utf-8-sig')


In [ ]:
import pandas as pd
from pyproj import Transformer

# 위경도 → EPSG:5181
transformer = Transformer.from_crs("EPSG:4326", "EPSG:5181", always_xy=True)

# 격자 크기 설정 
GRID_SIZE = 30  # 30m 격자

df = pd.read_csv("탄저병_sample_1.csv")  # 기존 결과물 불러오기
df = df.dropna(subset=["좌표-경도", "좌표-위도"])

# 위경도 → 5181 변환 및 격자 ID 생성
def generate_grid_id(lon, lat):
    x, y = transformer.transform(lon, lat)  # 위경도 → 평면좌표
    gx = int(x // GRID_SIZE)
    gy = int(y // GRID_SIZE)
    return f"{gx}_{gy}"

df["격자ID"] = df.apply(lambda row: generate_grid_id(row["좌표-경도"], row["좌표-위도"]), axis=1)

# 지점ID를 격자ID로 교체
df = df.drop(columns=["지점ID"])
df = df.rename(columns={"격자ID": "지점ID"})
cols = ['지점ID'] + [col for col in df.columns if col != '지점ID']
df = df[cols]

df = df.sort_values(by='지점ID')

# 6. 저장
df.to_csv("탄저병_sample_2.csv", index=False, encoding="utf-8-sig")


In [ ]:
import pandas as pd
import numpy as np
from pyproj import Transformer
from scipy.spatial import cKDTree  # pip install scipy

DISEASE_CSV = "탄저병_sample_2.csv"
AWS_CSV     = "/Users/doyoung-gil/Desktop/연구실/AIM_PEX_DT03_샘플데이터/KMA_AWS_DAILY.csv"

# 1) AWS 일자료 로드
usecols = ["stn_id","stn_nm","tm","lon","lat","ht","rn_day","tmax","tmin","wx_day"]
dtypes  = {
    "stn_id":"int32", "stn_nm":"string",
    "lon":"float32", "lat":"float32",
    "ht":"float32", "rn_day":"float32", "tmax":"float32","tmin":"float32",
    "wx_day":"float32" 
}
aws = pd.read_csv(AWS_CSV, usecols=usecols, dtype=dtypes, parse_dates=["tm"])
aws = aws.rename(columns={"tm":"date", "stn_nm":"stn_name"})

# 중복일 처리(안정성)
group_cols = ["stn_id","stn_name","date","lon","lat"]
agg = {"tmax":"mean","tmin":"mean","rn_day":"mean","ht":"mean","wx_day":"first"}
aws = aws.groupby(group_cols, as_index=False).agg(agg)

# 관측소 메타(좌표)
stn_meta = aws[["stn_id","stn_name","lon","lat"]].drop_duplicates("stn_id").reset_index(drop=True)

# 샘플 로드
df = pd.read_csv(DISEASE_CSV).dropna(subset=["좌표-경도","좌표-위도"]).copy()
df["좌표-경도"] = df["좌표-경도"].astype(float)
df["좌표-위도"] = df["좌표-위도"].astype(float)
df["date"] = pd.to_datetime(df["조사일자(YYYYMMDD)"].astype(str), format="%Y%m%d", errors="coerce")

# 좌표 변환(5181) & 최근접 관측소 매칭(거리 컬럼은 저장 안 함)
to_5181 = Transformer.from_crs("EPSG:4326","EPSG:5181", always_xy=True)
stn_xy = np.column_stack(to_5181.transform(stn_meta["lon"].values, stn_meta["lat"].values))
site_xy = np.column_stack(to_5181.transform(df["좌표-경도"].values, df["좌표-위도"].values))

idx = cKDTree(stn_xy).query(site_xy, k=1)[1]  # 인덱스만 사용
df["nearest_stn_id"]   = stn_meta.loc[idx, "stn_id"].values
df["nearest_stn_name"] = stn_meta.loc[idx, "stn_name"].values

# 날짜로 기상 붙이기
weather_cols = ["stn_id","date","ht","rn_day","tmax","tmin","wx_day"]
df_merged = df.merge(
    aws[weather_cols],
    left_on=["nearest_stn_id","date"],
    right_on=["stn_id","date"],
    how="left"
).drop(columns=["stn_id", "date"])  # date도 제거

# 조사일자 바로 뒤에 기상 컬럼 배치
wx_bundle = ["nearest_stn_id","nearest_stn_name","ht","rn_day","tmax","tmin","wx_day"]
cols = df_merged.columns.tolist()
pos = cols.index("조사구분")
head = cols[:pos+1]
used = set(head) | set(wx_bundle) | {"date"}  # 이미 쓴 것 + 지울 것
tail = [c for c in cols if c not in used]

df_merged = df_merged[head + wx_bundle + tail]

# 보기 좋게 정렬
if "지점ID" in df_merged.columns:
    df_merged = df_merged.sort_values(["지점ID","조사일자(YYYYMMDD)"]).reset_index(drop=True)

# 저장
df_merged.to_csv("탄저병_with_AWS_daily_1.csv", index=False, encoding="utf-8-sig")
print("완료 → 탄저병_with_AWS_daily_1.csv")
print(df_merged.head(3))


완료 → 탄저병_with_AWS_daily_!.csv
         지점ID  조사년도 지역-시도 지역-시군구 지역-읍면동     상세주소       좌표-경도      좌표-위도  조사회차  \
0  10000_6494  2016  경상남도    진주시    집현면  냉정리 377  128.098696  35.245024     1   
1  10000_6494  2016  경상남도    진주시    집현면  냉정리 377  128.098696  35.245024     1   
2  10000_6494  2016  경상남도    진주시    집현면  냉정리 377  128.098696  35.245024     2   

   조사일자(YYYYMMDD)  ...  ht rn_day  tmax tmin  wx_day  탄저병-병든과수/750과  \
0        20160701  ... NaN    NaN   NaN  NaN     NaN            NaN   
1        20160701  ... NaN    NaN   NaN  NaN     NaN            NaN   
2        20160718  ... NaN    NaN   NaN  NaN     NaN            NaN   

   탄저병-병든과율(%)  탄저병-병든꼬투리수(협/300협)  탄저병-병든주율(%)  탄저병-발병도  
0          NaN                 NaN          NaN      NaN  
1          NaN                 0.0          0.0      NaN  
2          NaN                 0.0          0.0      NaN  

[3 rows x 24 columns]


In [ ]:
# CSV 불러오기
df = pd.read_csv("탄저병_with_AWS_daily_1.csv", encoding="utf-8-sig")  # 필요시 encoding 변경


print(sorted(df["조사년도"].unique()))

[2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]


In [16]:
df = pd.read_csv("기상데이터/2023_WeatherData.csv", encoding="utf-8-sig")  # 필요시 encoding 변경

print(df["지점명"].unique())


['속초' '북춘천' '철원' '동두천' '파주' '대관령' '춘천' '백령도' '북강릉' '강릉' '동해' '서울' '인천'
 '원주' '울릉도' '수원' '영월' '충주' '서산' '울진' '청주' '대전' '추풍령' '안동' '상주' '포항' '군산'
 '대구' '전주' '울산' '창원' '광주' '부산' '통영' '목포' '여수' '흑산도' '완도' '고창' '순천' '홍성'
 '서청주' '제주' '고산' '성산' '서귀포' '진주' '강화' '양평' '이천' '인제' '홍천' '태백' '정선군' '제천'
 '보은' '천안' '보령' '부여' '금산' '세종' '부안' '임실' '정읍' '남원' '장수' '고창군' '영광군' '김해시'
 '순창군' '북창원' '양산시' '보성군' '강진군' '장흥' '해남' '고흥' '의령군' '함양군' '광양시' '진도군' '봉화'
 '영주' '문경' '청송군' '영덕' '의성' '구미' '영천' '경주시' '거창' '합천' '밀양' '산청' '거제' '남해'
 '북부산']


In [23]:
# 파일 경로
kma_file = "/Users/doyoung-gil/Desktop/연구실/AIM_PEX_DT03_샘플데이터/KMA_AWS_DAILY.csv"  # 교수님이 준 파일
new_file = "기상데이터/2023_WeatherData.csv"  # 방금 변환한 파일
new1_file = "/Users/doyoung-gil/Downloads/WeatherData.csv"

# 파일 불러오기 (UTF-8 기준)
kma_df = pd.read_csv(kma_file)
new_df = pd.read_csv(new_file)
new1_df = pd.read_csv(new1_file)


# 관측소 ID/이름 컬럼 이름 맞춰서 불러오기 (여기선 '지점' 가정)
kma_stations = set(kma_df['stn_nm'])
new_stations = set(new_df['지점명'])
new1_stations = set(new1_df['지점명'])

# KMA에는 있는데, 새 파일에 없는 관측소
missing_in_new = kma_stations - new_stations - new1_stations

print(f"새 파일에 없는 관측소 개수: {len(missing_in_new)}")
print("목록:", missing_in_new)

새 파일에 없는 관측소 개수: 248
목록: {'만장굴', '읍내', '신례', '포항(공) *', '군산(레)', '용평', '영중면', '풍양', '서청주(예)', '대구(공) *', '김포(공)', '안덕', '가남', '서구', '함양', '정선', '동천안', '하면', '백석읍', '창현', '서탄면', '선단동', '세종연기', '은현면', '청주(공) *', '덕정동', '일죽 *', '상북', '의령', '수리산길', '평도', '대야', '관인', '북내', '대곶', '이동', '인계', '가대암', '죽학', '신둔', '인천(공)', '세종(예)', '문덕', '홍산', '표선면', '아라', '송파 *', '남동공단', '양성', '영등포 *', '오전동 *', '봉성', '연기', '송산', '수택동', '경기', '북격렬비도 *', '안계', '원동', '장남', '외서', '홍북', '풍암 *', '간여암', '양산', '금사 *', '산북', '청원', '정연', '군남', '강문', '대진', '진도읍', '두촌', '설봉', '갈매여', '송탄', '오포', '남부', '모가', '모슬포', '웅상', '하봉암', '마장', '능곡', '대연', '회수', '장흥면', '처인역삼', '오성면', '여수(공)', '남촌 *', '하조도', '연천', '실촌 *', '대청', '화순북', '동래구', '북항 *', '몽탄', '금악', '영월주천', '화원', '우정', '북악산', '서초 *', '농암', '기상(연)', '원주백운산', '한라생태숲', '이덕서', '왕징', '심동리', '강현', '판교', '임회', '오남', '소하', '전의', '지귀도', '포승', '문경읍', '이동묵리', '수영만', '강진', '김해(공) *', '청산', '서신', '기흥구', '무등봉', '서호', '이원', '고덕면', '격렬 *', '선흘', '남항 *', '성거', '북항', '공촌동', '모악산', '가평', '경주',

In [20]:
df = pd.read_csv("/Users/doyoung-gil/Desktop/연구실/AIM_PEX_DT03_샘플데이터/KMA_AWS_DAILY.csv", encoding="utf-8-sig")  # 필요시 encoding 변경

kma_stations1 = set(kma_df['stn_nm'])

print(sorted(df["stn_nm"].unique()))
print(f"관측소 개수: {len(kma_stations1)}")


['가거도', '가곡', '가남', '가대암', '가덕', '가덕도', '가산', '가야산', '가파도', '가평', '가평북면', '가평조종', '간동', '간성', '간여암', '간절곶', '갈매여', '감포', '강남', '강동', '강릉', '강릉성산', '강릉왕산', '강문', '강북 *', '강서', '강정', '강진', '강진군', '강진면', '강현', '강화', '개천', '거문도', '거제', '거창', '격렬 *', '경기', '경기가평', '경기광주', '경산', '경서동', '경주', '경주시', '경포', '계룡', '계룡산', '고군', '고덕면', '고령', '고산', '고삼', '고성', '고양', '고양고봉', '고잔', '고창', '고창군', '고흥', '곡성', '공단', '공도', '공성', '공주', '공촌동', '과기원', '과천', '관산', '관악', '관악(레)', '관인', '광덕산', '광릉', '광명', '광명노온', '광산', '광안', '광양', '광양백운산', '광양시', '광양읍', '광주', '광주(공) *', '광주남구', '광진', '광탄', '괴산', '교동', '구례', '구로', '구룡령', '구룡포', '구리', '구미', '구이', '구좌', '군남', '군산', '군산(레)', '군산산단', '군위', '군포', '군포금정', '궁촌', '귀래', '근흥', '금강송', '금곡', '금남', '금사 *', '금산', '금악', '금왕', '금일', '금정구', '금천', '금촌', '기계', '기린', '기상(과)', '기상(연)', '기상청', '기장', '기흥구', '기흥구갈', '길곡', '길안', '김녕', '김제', '김천', '김포', '김포(공)', '김포장기', '김해(공) *', '김해시', '김화', '나로도', '나주', '낙월도', '낙천', '남동공단', '남면', '남방', '남부', '남산', '남양주', '남원', '남이섬', '남촌', '남촌 *', '남항